# 🏙️ Smart City — Traffic Pattern Forecasting
### Project 9 | Data Science for Urban Traffic Management
---
**Goal:** Forecast traffic volumes at 4 city junctions & provide infrastructure planning insights.

**How to run:**
1. Place `traffic.csv` in the same folder as this notebook
2. Run each cell top to bottom with `Shift + Enter`


## 📦 Step 0 — Install & Import Libraries

In [ ]:
# Install all required libraries
import sys
!{sys.executable} -m pip install pandas numpy matplotlib seaborn scikit-learn xgboost statsmodels plotly -q
print('✅ All libraries installed successfully!')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import timedelta

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX

import os
os.makedirs('plots', exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
print('✅ All imports done!')

## 📂 Step 1 — Load & Explore Data

In [ ]:
# ── Load CSV ──────────────────────────────────────────────────
df = pd.read_csv('traffic.csv')
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

# Parse datetime (handles both 'datetime' and 'date'+'time' columns)
if 'datetime' in df.columns:
    df['datetime'] = pd.to_datetime(df['datetime'])
elif 'date' in df.columns and 'time' in df.columns:
    df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'])
    df.drop(['date', 'time'], axis=1, inplace=True)

df.sort_values('datetime', inplace=True)
df.reset_index(drop=True, inplace=True)

print('━'*50)
print(f'  Rows       : {len(df):,}')
print(f'  Columns    : {list(df.columns)}')
print(f'  Date range : {df.datetime.min().date()} → {df.datetime.max().date()}')
print(f'  Junctions  : {sorted(df.junction.unique())}')
print(f'  Missing    : {df.isnull().sum().sum()}')
print('━'*50)
df.head(10)

In [ ]:
print('Statistical Summary:')
df.describe()

In [ ]:
# Handle missing values
df['vehicles'] = df['vehicles'].fillna(method='ffill')

# Traffic per junction summary
print('Average vehicles per junction:')
print(df.groupby('junction')['vehicles'].agg(['mean','min','max','std']).round(2))

## 🛠️ Step 2 — Feature Engineering

In [ ]:
# Public holidays list (add more as needed)
HOLIDAYS = [
    '2019-01-01','2019-01-14','2019-01-26','2019-03-04',
    '2019-03-21','2019-04-14','2019-04-17','2019-04-18',
    '2019-04-19','2019-05-18','2019-06-05','2019-08-15',
    '2019-10-02','2019-10-27','2019-11-10','2019-12-25',
]

dt = df['datetime']

# Temporal features
df['hour']        = dt.dt.hour
df['day_of_week'] = dt.dt.dayofweek           # 0=Monday
df['day_name']    = dt.dt.day_name()
df['month']       = dt.dt.month
df['month_name']  = dt.dt.month_name()
df['quarter']     = dt.dt.quarter
df['week']        = dt.dt.isocalendar().week.astype(int)
df['year']        = dt.dt.year

# Calendar flags
df['is_weekend']  = (dt.dt.dayofweek >= 5).astype(int)
df['is_holiday']  = dt.dt.normalize().isin(pd.to_datetime(HOLIDAYS)).astype(int)
df['is_peak']     = (dt.dt.hour.between(7, 9) | dt.dt.hour.between(17, 19)).astype(int)

# Cyclical encoding (avoids midnight/Jan discontinuity)
df['hour_sin']    = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos']    = np.cos(2 * np.pi * df['hour'] / 24)
df['dow_sin']     = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos']     = np.cos(2 * np.pi * df['day_of_week'] / 7)
df['month_sin']   = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos']   = np.cos(2 * np.pi * df['month'] / 12)

print('✅ Feature engineering done!')
print('New features:', ['hour','day_of_week','month','quarter','week','is_weekend',
                        'is_holiday','is_peak','hour_sin','hour_cos','dow_sin',
                        'dow_cos','month_sin','month_cos'])
df.head(3)

## 📊 Step 3 — Exploratory Data Analysis (EDA)

In [ ]:
junctions = sorted(df['junction'].unique())

# ── 3a. Traffic volume over time ──────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 10))
fig.suptitle('Hourly Traffic Volume per Junction', fontsize=16, fontweight='bold')

for ax, junc, col in zip(axes.flatten(), junctions, COLORS):
    sub = df[df['junction'] == junc]
    ax.plot(sub['datetime'], sub['vehicles'], color=col, linewidth=0.5, alpha=0.7)
    ax.set_title(f'Junction {junc}', fontsize=13)
    ax.set_xlabel('Date'); ax.set_ylabel('Vehicles / hour')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig('plots/01_traffic_volume.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: plots/01_traffic_volume.png')

In [ ]:
# ── 3b. Average traffic by hour of day ───────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Average Vehicles by Hour of Day', fontsize=15, fontweight='bold')

for ax, junc, col in zip(axes.flatten(), junctions, COLORS):
    sub = df[df['junction'] == junc]
    hourly = sub.groupby('hour')['vehicles'].mean()
    ax.bar(hourly.index, hourly.values, color=col, alpha=0.85)
    ax.axvspan(7, 9, alpha=0.12, color='red', label='Morning peak (7-9 AM)')
    ax.axvspan(17, 19, alpha=0.12, color='orange', label='Evening peak (5-7 PM)')
    ax.set_title(f'Junction {junc}'); ax.set_xlabel('Hour'); ax.set_ylabel('Avg Vehicles')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('plots/02_hourly_pattern.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: plots/02_hourly_pattern.png')

In [ ]:
# ── 3c. Day of week pattern ───────────────────────────────────
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Average Vehicles by Day of Week', fontsize=15, fontweight='bold')

for ax, junc, col in zip(axes.flatten(), junctions, COLORS):
    sub = df[df['junction'] == junc]
    daily = sub.groupby('day_of_week')['vehicles'].mean()
    bars = ax.bar(days, daily.values, color=col, alpha=0.85)
    for b in bars[5:]: b.set_color('#aaaaaa')    # grey weekends
    ax.set_title(f'Junction {junc}'); ax.set_xlabel('Day'); ax.set_ylabel('Avg Vehicles')

plt.tight_layout()
plt.savefig('plots/03_weekly_pattern.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: plots/03_weekly_pattern.png')

In [ ]:
# ── 3d. Monthly trend ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
for junc, col in zip(junctions, COLORS):
    sub = df[df['junction'] == junc]
    monthly = sub.groupby(['month', 'month_name'])['vehicles'].mean().reset_index()
    monthly.sort_values('month', inplace=True)
    ax.plot(monthly['month_name'], monthly['vehicles'],
            marker='o', label=f'Junction {junc}', color=col, linewidth=2)

ax.set_title('Monthly Average Traffic per Junction', fontsize=14, fontweight='bold')
ax.set_xlabel('Month'); ax.set_ylabel('Avg Vehicles / hour')
ax.legend(); plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('plots/04_monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: plots/04_monthly_trend.png')

In [ ]:
# ── 3e. Holiday vs Normal Day ─────────────────────────────────
df['day_type'] = df['is_holiday'].map({0: 'Normal Day', 1: 'Holiday'})

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df, x='junction', y='vehicles', hue='day_type',
            palette={'Normal Day': '#1f77b4', 'Holiday': '#d62728'}, ax=ax)
ax.set_title('Traffic Volume: Holiday vs Normal Days (per Junction)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Junction'); ax.set_ylabel('Vehicles / hour')
plt.tight_layout()
plt.savefig('plots/05_holiday_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: plots/05_holiday_comparison.png')

In [ ]:
# ── 3f. Correlation heatmap ───────────────────────────────────
num_cols = ['vehicles', 'hour', 'day_of_week', 'month',
            'is_weekend', 'is_holiday', 'is_peak']
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/06_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: plots/06_correlation.png')

## 📉 Step 4 — Seasonal Decomposition

In [ ]:
# Decompose daily traffic for all junctions
fig, big_axes = plt.subplots(4, 4, figsize=(22, 20))
fig.suptitle('Seasonal Decomposition — All Junctions (Daily)', fontsize=16, fontweight='bold')

component_titles = ['Observed', 'Trend', 'Seasonality', 'Residual']

for j_idx, junc in enumerate(junctions):
    sub   = df[df['junction'] == junc].copy()
    daily = sub.set_index('datetime')['vehicles'].resample('D').mean().dropna()
    result = seasonal_decompose(daily, model='additive', period=7)
    components = [daily, result.trend, result.seasonal, result.resid]

    for c_idx, (comp, title) in enumerate(zip(components, component_titles)):
        ax = big_axes[j_idx][c_idx]
        ax.plot(comp, linewidth=0.9, color=COLORS[j_idx])
        if j_idx == 0: ax.set_title(title, fontsize=12, fontweight='bold')
        if c_idx == 0: ax.set_ylabel(f'Junction {junc}', fontsize=11, fontweight='bold')
        ax.tick_params(axis='x', rotation=30, labelsize=7)

plt.tight_layout()
plt.savefig('plots/07_seasonal_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: plots/07_seasonal_decomposition.png')

## 🤖 Step 5 — ML Model Training & Evaluation

In [ ]:
FEATURE_COLS = [
    'hour', 'day_of_week', 'month', 'quarter', 'week',
    'is_weekend', 'is_holiday', 'is_peak',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'month_sin', 'month_cos', 'junction'
]

X = df[FEATURE_COLS]
y = df['vehicles']

# Time-ordered split (no shuffle — respects temporal order)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

print(f'Training samples : {len(X_train):,}')
print(f'Testing samples  : {len(X_test):,}')
print(f'Features used    : {FEATURE_COLS}')

In [ ]:
# Define models
models = {
    'Random Forest' : RandomForestRegressor(
        n_estimators=200, max_depth=12, n_jobs=-1, random_state=42),
    'Gradient Boost': GradientBoostingRegressor(
        n_estimators=300, max_depth=5, learning_rate=0.05, random_state=42),
    'XGBoost'       : xgb.XGBRegressor(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0),
}

results, preds = [], {}
print('Training models...\n' + '─'*55)

for name, model in models.items():
    print(f'  Fitting {name}...')
    model.fit(X_train, y_train)
    p = model.predict(X_test)
    preds[name] = p
    mae  = mean_absolute_error(y_test, p)
    rmse = np.sqrt(mean_squared_error(y_test, p))
    r2   = r2_score(y_test, p)
    results.append({'Model': name, 'MAE': round(mae,2), 'RMSE': round(rmse,2), 'R²': round(r2,4)})
    print(f'    MAE={mae:.2f}  RMSE={rmse:.2f}  R²={r2:.4f}')

res_df = pd.DataFrame(results)
print('\n' + '─'*55)
print(res_df.to_string(index=False))

In [ ]:
# Model comparison chart
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Model Comparison — MAE / RMSE / R²', fontsize=14, fontweight='bold')

for ax, metric, col in zip(axes, ['MAE','RMSE','R²'], ['#e74c3c','#e67e22','#27ae60']):
    bars = ax.bar(res_df['Model'], res_df[metric], color=col, alpha=0.85)
    ax.set_title(metric, fontsize=13); ax.set_ylabel(metric)
    for bar, val in zip(bars, res_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()*0.97,
                str(val), ha='center', va='top', fontsize=10, color='white', fontweight='bold')
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=15)

plt.tight_layout()
plt.savefig('plots/08_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: plots/08_model_comparison.png')

In [ ]:
# Actual vs Predicted — XGBoost
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(y_test.values[:400], label='Actual', linewidth=1.5, color='#1f77b4')
ax.plot(preds['XGBoost'][:400], label='XGBoost Predicted',
        linestyle='--', linewidth=1.5, alpha=0.9, color='#d62728')
ax.set_title('Actual vs Predicted Traffic — XGBoost (First 400 Test Samples)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Sample Index'); ax.set_ylabel('Vehicles / hour'); ax.legend()
plt.tight_layout()
plt.savefig('plots/09_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: plots/09_actual_vs_predicted.png')

In [ ]:
# Feature importance
best_model = models['XGBoost']
imp = pd.Series(best_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
imp.plot(kind='barh', color='#3498db', alpha=0.85, ax=ax)
ax.set_title('XGBoost Feature Importance', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('plots/10_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: plots/10_feature_importance.png')

## 📆 Step 6 — SARIMA Time-Series Forecast

In [ ]:
FORECAST_DAYS = 7

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle(f'SARIMA {FORECAST_DAYS}-Day Traffic Forecast — All Junctions',
             fontsize=15, fontweight='bold')

for ax, junc, col in zip(axes.flatten(), junctions, COLORS):
    sub   = df[df['junction'] == junc]
    daily = sub.set_index('datetime')['vehicles'].resample('D').mean().dropna()
    train = daily[:-FORECAST_DAYS]
    test  = daily[-FORECAST_DAYS:]

    sarima_model = SARIMAX(train, order=(1,1,1), seasonal_order=(1,1,1,7),
                            enforce_stationarity=False, enforce_invertibility=False)
    sarima_fit   = sarima_model.fit(disp=False)
    forecast     = sarima_fit.forecast(steps=FORECAST_DAYS)

    future_dates = pd.date_range(train.index[-1] + timedelta(days=1),
                                  periods=FORECAST_DAYS, freq='D')
    fc_series = pd.Series(forecast.values, index=future_dates)

    mae  = mean_absolute_error(test.values, forecast.values)
    rmse = np.sqrt(mean_squared_error(test.values, forecast.values))
    print(f'Junction {junc} → MAE={mae:.2f}  RMSE={rmse:.2f}')

    ax.plot(daily[-45:], label='Historical (last 45 days)', color=col, linewidth=2)
    ax.plot(fc_series,   label='Forecast', color='#d62728', linestyle='--',
            linewidth=2, marker='o')
    ax.plot(test,        label='Actual', color='#2ca02c', linewidth=1.5, alpha=0.8)
    ax.set_title(f'Junction {junc}  |  MAE={mae:.1f}'); ax.legend(fontsize=8)
    ax.set_xlabel('Date'); ax.set_ylabel('Avg Vehicles / day')
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig('plots/11_sarima_forecast.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: plots/11_sarima_forecast.png')

## 🚦 Step 7 — Peak Hour Analysis & Infrastructure Insights

In [ ]:
# Heatmap: Hour × Day of Week per junction
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Traffic Intensity Heatmap — Hour of Day × Day of Week',
             fontsize=15, fontweight='bold')

for ax, junc in zip(axes.flatten(), junctions):
    sub   = df[df['junction'] == junc]
    pivot = sub.pivot_table(values='vehicles', index='hour',
                            columns='day_of_week', aggfunc='mean')
    pivot.columns = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
    sns.heatmap(pivot, cmap='YlOrRd', ax=ax, linewidths=0.3, cbar=True, annot=False)
    ax.set_title(f'Junction {junc}', fontsize=13)
    ax.set_xlabel('Day of Week'); ax.set_ylabel('Hour of Day')

plt.tight_layout()
plt.savefig('plots/12_peak_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: plots/12_peak_heatmap.png')

In [ ]:
# ── Per-junction infrastructure recommendations ───────────────
print('\n' + '='*65)
print('         SMART CITY — INFRASTRUCTURE RECOMMENDATIONS')
print('='*65)

for junc in junctions:
    sub          = df[df['junction'] == junc]
    busiest_hour = sub.groupby('hour')['vehicles'].mean().idxmax()
    busiest_day  = sub.groupby('day_name')['vehicles'].mean().idxmax()
    quietest_hr  = sub.groupby('hour')['vehicles'].mean().idxmin()
    peak_vol     = sub.groupby('hour')['vehicles'].mean().max()
    avg_vol      = sub['vehicles'].mean()
    wknd_avg     = sub[sub['is_weekend']==1]['vehicles'].mean()
    wkday_avg    = sub[sub['is_weekend']==0]['vehicles'].mean()
    h_diff       = sub.groupby('is_holiday')['vehicles'].mean().diff().iloc[-1]

    print(f'\n🔷 Junction {junc}')
    print(f'   Busiest hour       : {busiest_hour:02d}:00 — {busiest_hour+1:02d}:00')
    print(f'   Busiest day        : {busiest_day}')
    print(f'   Quietest hour      : {quietest_hr:02d}:00')
    print(f'   Peak volume        : {peak_vol:.0f} vehicles/hr')
    print(f'   Weekday avg        : {wkday_avg:.1f} vs Weekend avg: {wknd_avg:.1f}')
    print(f'   Holiday effect     : {h_diff:+.1f} vehicles/hr vs normal days')
    print(f'   Recommendation     : Extend green signal timing at {busiest_hour:02d}:00–{busiest_hour+1:02d}:00')
    print(f'                        Schedule road maintenance at {quietest_hr:02d}:00')

print('\n' + '='*65)
print('\n✅ Project complete! All plots saved to /plots/ folder.')